# DOAgent — Grid-world demo

This notebook runs the **grid-world** scenario step by step: multiple agents explore a grid with partial observations and share discovered cells via DOAgent. It only uses the `doagent` library and code defined here. Run in Google Colab.

**What you'll do:** Install doagent → define a minimal grid env and policy → create a file-backed session → run the loop (with shared map) → run analysis including causal attribution.

## Step 1 — Install the library

Install DOAgent from the repository. No PettingZoo needed for this demo.

In [ ]:
!pip install -q git+https://github.com/cabrerac/doagent.git

## Step 2 — Imports

Import Session, make_env, RunReporter, and the analysis modules.

In [ ]:
import random
from typing import Any, Dict, List, Tuple

from doagent import Session, RunReporter, make_env
from doagent.analysis import accountability, interpretability, provenance, traceability

## Step 3 — Define a minimal grid-world environment

The env must provide:
- `reset(seed=...)` → observations dict (per agent) with `position`, `cells`, `width`, `height`
- `step(actions)` → dict with `observations`, `rewards`, `terminations`
- `agents` → list of agent ids

Actions: 0=stay, 1=left, 2=right, 3=up, 4=down. Reward = number of newly discovered cells for that agent.

In [ ]:
class SimpleGridEnv:
    def __init__(self, width=4, height=4, agent_ids=None, landmarks=1, observation_radius=1, max_cycles=15, seed=None):
        self._w, self._h = width, height
        self._agent_ids = agent_ids or ["agent_0", "agent_1"]
        self._landmarks = landmarks
        self._radius = observation_radius
        self._max_cycles = max_cycles
        self._rng = random.Random(seed)
        self._positions = {}
        self._landmark_positions = []
        self._discovered = set()
        self._step_count = 0

    @property
    def agents(self):
        return list(self._agent_ids)

    def _rand_pos(self):
        return (self._rng.randrange(self._w), self._rng.randrange(self._h))

    def _observe(self, agent_id):
        x, y = self._positions[agent_id]
        cells = []
        for dx in range(-self._radius, self._radius + 1):
            for dy in range(-self._radius, self._radius + 1):
                cx, cy = x + dx, y + dy
                if 0 <= cx < self._w and 0 <= cy < self._h:
                    value = "landmark" if (cx, cy) in self._landmark_positions else "empty"
                    cells.append({"x": cx, "y": cy, "value": value})
        return {"position": {"x": x, "y": y}, "cells": cells, "width": self._w, "height": self._h}

    def _move(self, pos, action):
        x, y = pos
        if action == 1: x -= 1
        elif action == 2: x += 1
        elif action == 3: y += 1
        elif action == 4: y -= 1
        return (max(0, min(self._w - 1, x)), max(0, min(self._h - 1, y)))

    def reset(self, *, seed=None):
        if seed is not None:
            self._rng.seed(seed)
        self._step_count = 0
        self._positions = {aid: self._rand_pos() for aid in self._agent_ids}
        self._landmark_positions = [self._rand_pos() for _ in range(self._landmarks)]
        self._discovered.clear()
        obs = {aid: self._observe(aid) for aid in self._agent_ids}
        for o in obs.values():
            for c in o["cells"]:
                self._discovered.add((c["x"], c["y"]))
        return obs

    def step(self, actions):
        self._step_count += 1
        for aid, a in actions.items():
            if aid in self._positions:
                self._positions[aid] = self._move(self._positions[aid], a)
        obs = {aid: self._observe(aid) for aid in self._agent_ids}
        rewards = {}
        for aid, o in obs.items():
            new_cells = sum(1 for c in o["cells"] if (c["x"], c["y"]) not in self._discovered)
            for c in o["cells"]:
                self._discovered.add((c["x"], c["y"]))
            rewards[aid] = float(new_cells)
        done = self._step_count >= self._max_cycles
        return {"observations": obs, "rewards": rewards, "terminations": {aid: done for aid in self._agent_ids}}

def create_grid_env(*, width=4, height=4, agent_ids=None, landmarks=1, observation_radius=1, max_cycles=15, seed=None):
    return SimpleGridEnv(width=width, height=height, agent_ids=agent_ids, landmarks=landmarks,
                         observation_radius=observation_radius, max_cycles=max_cycles, seed=seed)

## Step 4 — Define a simple policy and build_shared_map

Agents need a policy that takes observation + shared map. We define a random explorer and a helper that turns recorded agent_update payloads into a merged map of cells (so agents can see what others have discovered).

In [ ]:
def _known_cells(shared_map):
    return {(c.get("x"), c.get("y")) for c in shared_map.get("cells", []) if c.get("x") is not None and c.get("y") is not None}

def random_explore_policy(params):
    rng = random.Random(params.get("seed", 0))
    def decide(request):
        obs = request.get("inputs", {}).get("observation", {})
        shared_map = request.get("inputs", {}).get("shared_map", {})
        pos = obs.get("position", {})
        x, y = int(pos.get("x", 0)), int(pos.get("y", 0))
        w, h = obs.get("width", 0), obs.get("height", 0)
        known = _known_cells(shared_map)
        neighbors = [(x-1,y,1), (x+1,y,2), (x,y+1,3), (x,y-1,4)]
        valid = [a for (nx,ny,a) in neighbors if 0<=nx<w and 0<=ny<h]
        unknown = [a for (nx,ny,a) in neighbors if 0<=nx<w and 0<=ny<h and (nx,ny) not in known]
        action = rng.choice(unknown) if unknown else (rng.choice(valid) if valid else 0)
        return {"decision": {"action": action}}
    return decide

def build_shared_map(records):
    cells = {}
    for r in records:
        payload = getattr(r, "payload", r) if not isinstance(r, dict) else r.get("payload", {})
        for c in (payload.get("local_knowledge", {}).get("observation", {}).get("cells", []) or payload.get("cells", [])):
            if c.get("x") is not None and c.get("y") is not None:
                cells[(c["x"], c["y"])] = c.get("value", "unknown")
    return {"cells": [{"x": x, "y": y, "value": v} for (x, y), v in cells.items()]}

## Step 5 — Create a file-backed session

Use file storage with `scenario_name` so the library creates a run folder. Register the policy and create the grid env.

In [ ]:
output_base = "./output"
agent_ids = ["agent_0", "agent_1"]
configs = [
    {"id": "agent_0", "policy": {"name": "random_explore", "params": {"seed": 1}}},
    {"id": "agent_1", "policy": {"name": "random_explore", "params": {"seed": 2}}},
]
session = Session.from_config({
    "shared_data": {"type": "file"},
    "scenario_name": "gridworld",
    "output_base": output_base,
    "run_config": {"logging_level": 2},
    "topology": {"mode": "centralised"},
    "policies": {"random_explore": random_explore_policy},
})
env = make_env(create_grid_env, width=4, height=4, agent_ids=agent_ids, landmarks=1, observation_radius=1, max_cycles=12, seed=0)
print(f"Run id: {session.run_id}")

## Step 6 — Wrap env, create agents, run the loop

Wrap the env with the session, create agents with goal `map_discovery` and `payload_type="map_update"`. Each round: get visible records, build shared map, let each agent decide with observation + shared_map, then step. Recording is automatic.

In [ ]:
wrapped = session.wrap_env(env, env_actor="gridworld_env")
agents = session.create_agents(configs, goal="map_discovery", payload_type="map_update")
observations = wrapped.reset(seed=42)
rounds = 12
for round_id in range(1, rounds + 1):
    actions = {}
    for aid in sorted(agents):
        shared_records = session.visible_records(aid, kind="agent_update")
        shared_map = build_shared_map(shared_records)
        result = agents[aid].decide(observations[aid], round_id, inputs={"observation": observations[aid], "shared_map": shared_map})
        actions[aid] = result["action"]
    step = wrapped.step(actions)
    observations = step["observations"]
print("Run completed.")

## Step 7 — Run analysis (provenance, traceability, accountability, interpretability)

Call each analysis module with `write_output=True`. The library writes PNG/PDF and JSON under `output/<run_id>/analysis/`. Grid-world has a discovery semantics so we include **accountability** (causal attribution). Use the effective id from provenance for interpretability.

In [ ]:
run_id = session.run_id
effective_id = provenance.render_chain_tree("last", run_id, output_base=output_base, write_output=True)
traceability.build_trace_graph(run_id, output_base=output_base, write_output=True)
accountability.causal_attribution(run_id, output_base=output_base, write_output=True)
last_id = effective_id or "last"
interpretability.get_explanations_for(last_id, run_id, output_base=output_base, write_output=True)
print(f"Analysis written to {output_base}/{run_id}/analysis/")
print("  - provenance/: provenance_tree.png, .pdf")
print("  - traceability/: trace_graph.png, .pdf")
print("  - accountability/: causal_attribution.png, .pdf")
print("  - interpretability/: explanations_for_last.json")

## Step 8 — View and interpret the analysis results

The analysis step wrote figures and JSON under `output/<run_id>/analysis/`. Below we display them and briefly explain what each shows.

- **Provenance tree** (`provenance/`): Chain of records that led to the last outcome—which decisions and env steps produced the final state.
- **Trace graph** (`traceability/`): Cause–effect links between records (who acted, what they observed, what changed).
- **Causal attribution** (`accountability/`): For discovery-style runs, which agents discovered which cells, and how much each agent contributed (productive vs redundant decisions). Useful to attribute “who found what.”
- **Explanations** (`interpretability/`): Records that explain the chosen outcome (e.g. the last step).

In [ ]:
from pathlib import Path
import json
from IPython.display import Image, display

base = Path(output_base) / run_id / "analysis"

# Provenance tree
provenance_png = base / "provenance" / "provenance_tree.png"
if provenance_png.exists():
    print("### Provenance tree")
    display(Image(filename=str(provenance_png)))
else:
    print("Provenance tree not found (re-run Step 7).")

# Trace graph
trace_png = base / "traceability" / "trace_graph.png"
if trace_png.exists():
    print("### Trace graph")
    display(Image(filename=str(trace_png)))
else:
    print("Trace graph not found (re-run Step 7).")

# Causal attribution (grid-world discovery)
attr_png = base / "accountability" / "causal_attribution.png"
if attr_png.exists():
    print("### Causal attribution")
    display(Image(filename=str(attr_png)))
else:
    print("Causal attribution figure not found (re-run Step 7).")

# Explanations JSON
expl_path = base / "interpretability" / "explanations_for_last.json"
if expl_path.exists():
    print("### Explanations for last outcome (excerpt)")
    with open(expl_path) as f:
        data = json.load(f)
    print(f"Total records: {len(data)}")
    if data:
        rec = data[0] if isinstance(data[0], dict) else data
        print("First record keys:", list(rec.keys()) if isinstance(rec, dict) else type(rec))
        print(json.dumps(rec if isinstance(rec, dict) else data[:1], indent=2)[:800] + ("..." if len(json.dumps(data)) > 800 else ""))
else:
    print("Explanations file not found (re-run Step 7).")

---
You can download the generated files from Colab (e.g. from the file browser) or inspect `output/<run_id>/analysis/` in your runtime.